# MRI/fMRI foundations: from NIfTI to experimental BOLD predictors

**NeuroAI Research Track · Wakeman & Henson · OpenNeuro `ds000117` · `sub-01 / ses-mri / run-01`**

Этот notebook объединяет два последовательных этапа исследования:

**Часть I. Пространственная организация нейровизуализационных данных**

`BIDS → NIfTI → voxel / volume / TR → affine → orientation → FOV → mean EPI → coregistration → visual QC`

**Часть II. Временная структура эксперимента и ожидаемый гемодинамический ответ**

`events.tsv → stimulus categories / repetition → event timelines → canonical HRF → convolution → faces vs scrambled predictors`

Основная задача: понять происхождение пространственных и временных данных, прежде чем приступать к *design matrix* и GLM.

> **Научные ограничения.** В части I мы показываем учебную регистрацию raw mean EPI к T1w без полного preprocessing (включая motion / distortion correction). В части II рассчитываем **модельные предикторы**, а не измеряем реальные BOLD-ответы на категории стимулов. Статистического GLM и выводов об активации здесь пока нет.

> **Данные и воспроизводимость.** Исходные MRI и `events.tsv` загружаются из публичного OpenNeuro S3 в локальную временную директорию, а не в репозиторий. Ячейки рассчитаны на последовательное выполнение сверху вниз в Google Colab.

## 1. Подготовка среды

Все зависимости устанавливаем **один раз**. В Google Colab сохраняем совместимые с его средой версии `pandas` и `requests`: ранее отдельная установка `nilearn` вызывала конфликт зависимостей. Если пакеты уже были импортированы до установки, перезапусти kernel перед выполнением остальных ячеек. При запуске на другой платформе проверь совместимость закреплённых версий.

In [ ]:
# Единая установка зависимостей вместо нескольких повторных %pip-команд.
# Nilearn 0.14.1 использует аргумент t_r (не tr) у функции spm_hrf().
%pip -q install "nibabel" "boto3" "dipy" "nilearn==0.14.1" "pandas==2.2.3" "requests==2.32.4"

In [ ]:
# Импорты для обеих частей notebook. Не повторяем их в последующих разделах.
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

from botocore import UNSIGNED
from botocore.config import Config
from matplotlib.lines import Line2D
from nibabel.affines import apply_affine
from nibabel.orientations import aff2axcodes
from nilearn.glm.first_level.hemodynamic_models import spm_hrf

In [ ]:
# В Colab используем /content; локально сохраняем временные данные внутри ../data/tmp.
DATA_DIR = (
    Path("/content/neuro_ai_data")
    if Path("/content").exists()
    else Path("../data/tmp")
)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Temporary data directory:", DATA_DIR)

## 2. OpenNeuro / BIDS: находим функциональные файлы

OpenNeuro публикует этот dataset через публичный S3 bucket.  
Подключаемся анонимно и смотрим содержимое `sub-01/ses-mri/func`.

Это одновременно показывает BIDS-структуру реального исследования:
`subject → session → modality → files`.

In [ ]:
# Создаём анонимный S3 client: credentials для публичного OpenNeuro bucket не нужны.
s3 = boto3.client(
    "s3",
    config=Config(signature_version=UNSIGNED),
)

bucket = "openneuro.org"
func_prefix = "ds000117/sub-01/ses-mri/func/"

# Получаем список файлов функциональной MRI-сессии sub-01.
response = s3.list_objects_v2(
    Bucket=bucket,
    Prefix=func_prefix,
)

func_keys = [obj["Key"] for obj in response.get("Contents", [])]

# Для компактности показываем только run-01.
for key in func_keys:
    if "run-01" in key:
        print(key)

Для одного `run-01` нас интересуют три BIDS-файла:

- `*_bold.nii.gz` - 4D измеренный fMRI BOLD-сигнал;
- `*_bold.json` - метаданные сканирования (TR и другие параметры);
- `*_events.tsv` - времена стимулов и поведенческих ответов.

Сначала работаем с пространственной структурой NIfTI, а во второй части анализируем экспериментальные события. JSON перечислен как часть BIDS, но отдельного разбора sidecar metadata в этой версии нет.

In [ ]:
# Явно задаём путь к run-01 BOLD, чтобы notebook не зависел от порядка файлов в S3 listing.
bold_key = (
    "ds000117/sub-01/ses-mri/func/"
    "sub-01_ses-mri_task-facerecognition_run-01_bold.nii.gz"
)

bold_path = DATA_DIR / "sub-01_run-01_bold.nii.gz"

# Скачиваем только один BOLD run, а не весь dataset.
if not bold_path.exists():
    s3.download_file(bucket, bold_key, str(bold_path))

print("BOLD file:", bold_path)

## 3. Загружаем BOLD NIfTI

`nib.load()` возвращает NIfTI image object.  
На этом этапе данные ещё не обязаны полностью загружаться в RAM: NiBabel может работать через proxy.

In [ ]:
# Загружаем 4D BOLD NIfTI.
func_img = nib.load(bold_path)

print("fMRI shape:", func_img.shape)
print("Voxel sizes + TR:", func_img.header.get_zooms())
print("Orientation:", aff2axcodes(func_img.affine))

Для `run-01` ожидаем:

- shape: `(64, 64, 33, 208)`;
- spatial voxel size: примерно `3 × 3 × 3.75 mm`;
- `TR = 2 s`.

То есть один volume — это `64 × 64 × 33`, а весь fMRI run — последовательность из 208 таких volumes.

In [ ]:
# Берём только первый 3D volume через data proxy.
# Так мы не загружаем весь 4D run в память только ради первой визуализации.
first_volume = np.asarray(func_img.dataobj[..., 0], dtype=np.float32)

print("First volume shape:", first_volume.shape)

In [ ]:
def plot_orthogonal_slices(volume, title_prefix):
    """Показывает центральные sagittal, coronal и axial срезы 3D volume."""
    x, y, z = (np.array(volume.shape) // 2).astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))

    # Axis 0 соответствует left-right направлению после учёта ориентации изображения.
    axes[0].imshow(volume[x, :, :].T, cmap="gray", origin="lower")
    axes[0].set_title(f"{title_prefix} — Sagittal")

    axes[1].imshow(volume[:, y, :].T, cmap="gray", origin="lower")
    axes[1].set_title(f"{title_prefix} — Coronal")

    axes[2].imshow(volume[:, :, z].T, cmap="gray", origin="lower")
    axes[2].set_title(f"{title_prefix} — Axial")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_orthogonal_slices(first_volume, "BOLD volume 0")

## 4. Structural T1w MRI

Теперь загружаем структурное T1w изображение **того же участника**.

T1w даёт гораздо более детальную анатомию, чем EPI/BOLD, поэтому позже будет нашим
`static / fixed image` для coregistration.

In [ ]:
# Известный BIDS-путь к T1w sub-01.
t1w_key = "ds000117/sub-01/ses-mri/anat/sub-01_ses-mri_T1w.nii.gz"
t1w_path = DATA_DIR / "sub-01_T1w.nii.gz"

if not t1w_path.exists():
    s3.download_file(bucket, t1w_key, str(t1w_path))

anat_img = nib.load(t1w_path)

print("T1w shape:", anat_img.shape)
print("T1w voxel sizes:", anat_img.header.get_zooms()[:3])
print("T1w orientation:", aff2axcodes(anat_img.affine))

In [ ]:
# Загружаем T1w как float32: этого достаточно для нашей визуализации и registration,
# а памяти требуется меньше, чем для float64.
anat_data = anat_img.get_fdata(dtype=np.float32)

plot_orthogonal_slices(anat_data, "T1w")

### T1w vs BOLD

Здесь важно не путать **размер массива** и **физическое разрешение**.

- T1w: примерно `1 × 1 × 1 mm` voxels;
- fMRI/EPI: примерно `3 × 3 × 3.75 mm` voxels.

Более мелкая voxel grid T1w содержит больше пространственной детализации.  
Resampling fMRI на T1w grid позднее **не создаст** эту потерянную детализацию.

## 5. Affine: voxel coordinates ↔ world coordinates

Voxel index — это адрес внутри NumPy-массива.  
Affine связывает этот индекс с физическими координатами изображения, обычно в миллиметрах.

Проверим одну точку `fMRI voxel = [30, 30, 15]`.

In [ ]:
print("fMRI affine:")
print(func_img.affine)

print("\nT1w affine:")
print(anat_img.affine)

In [ ]:
# Выбираем один voxel в исходной fMRI grid.
func_voxel = np.array([30, 30, 15])

# Переводим voxel index fMRI в физические world coordinates.
world_coord = apply_affine(func_img.affine, func_voxel)

# Через inverse affine T1w находим, где та же world coordinate лежит в T1w grid.
anat_voxel = apply_affine(
    np.linalg.inv(anat_img.affine),
    world_coord,
)

print("fMRI voxel:", func_voxel)
print("World coordinates, mm:", world_coord)
print("T1w voxel coordinate:", anat_voxel)
print("Nearest T1w voxel:", np.round(anat_voxel).astype(int))

Ключевое ограничение:

> Одинаковая **header-derived world coordinate** в двух изображениях ещё не доказывает,
> что их анатомическое содержимое идеально совмещено.

Affine описывает геометрию файлов.  
**Registration** оценивает дополнительное преобразование по самому изображению.

## 6. Orientation и canonical RAS

Проверяем направления осей, зашитые в affine.

In [ ]:
print("Original fMRI orientation:", aff2axcodes(func_img.affine))
print("Original T1w orientation:", aff2axcodes(anat_img.affine))

В этих данных исходные orientations различаются:

- fMRI: `LAS`;
- T1w: `RAS`.

Это объясняет, почему одинаковые array indices могут визуально находиться с разных сторон.

Canonical orientation — это **переупорядочивание / flipping voxel axes**, а не registration.

In [ ]:
# Приводим оба изображения к ближайшей canonical (RAS+) ориентации.
# Это удобно для обучения и визуального сравнения, но НЕ является обязательным шагом
# registration: корректный affine уже несёт информацию об ориентации.
func_can = nib.as_closest_canonical(func_img)
anat_can = nib.as_closest_canonical(anat_img)

print("Canonical fMRI:", aff2axcodes(func_can.affine))
print("Canonical T1w:", aff2axcodes(anat_can.affine))

In [ ]:
# World coordinate не меняется от reorientation.
# Меняется только voxel index, которым эта физическая точка адресуется в новой grid.
func_voxel_can = apply_affine(
    np.linalg.inv(func_can.affine),
    world_coord,
)
anat_voxel_can = apply_affine(
    np.linalg.inv(anat_can.affine),
    world_coord,
)

print("Canonical fMRI voxel:", func_voxel_can)
print("Canonical T1w voxel:", anat_voxel_can)
print("Same world coordinate:", world_coord)

Итого:

`voxel index → affine → world coordinate`

а orientation говорит, в каких анатомических направлениях идут оси массива.

`canonical orientation ≠ registration`

## 7. Field of View (FOV)

Сравним approximate physical coverage вдоль voxel axes.

Здесь используем:

`number of voxels × voxel spacing`

Это полезная оценка acquisition coverage, но не мера «масштаба мозга».

In [ ]:
func_voxel_sizes = np.array(func_can.header.get_zooms()[:3], dtype=float)
anat_voxel_sizes = np.array(anat_can.header.get_zooms()[:3], dtype=float)

func_fov = np.array(func_can.shape[:3]) * func_voxel_sizes
anat_fov = np.array(anat_can.shape[:3]) * anat_voxel_sizes

print("fMRI voxel sizes, mm:", func_voxel_sizes)
print("T1w voxel sizes, mm:", anat_voxel_sizes)
print("Approx. fMRI FOV, mm:", func_fov)
print("Approx. T1w FOV, mm:", anat_fov)

Для этого run fMRI покрывает меньший физический объём, особенно по одной из spatial axes.

Это **limited FOV / coverage**, а не доказательство того, что мозг на fMRI «меньше».

Дополнительно EPI чувствительна к susceptibility artifacts около границ
`brain ↔ bone ↔ air`, особенно в orbitofrontal области. Возможны:

- geometric distortion;
- signal dropout.

Rigid registration не способна восстановить потерянный signal.

## 8. Mean EPI для coregistration

Для регистрации 4D run к T1w нужен один репрезентативный 3D functional image.

В учебном примере берём mean EPI: усредняем каждый voxel по времени.

> Ограничение: в реальном preprocessing обычно сначала учитывают head motion.
> Здесь mean EPI строится без motion correction только для понимания registration.

In [ ]:
# Загружаем canonical 4D fMRI как float32 и усредняем по temporal axis.
func_can_data = func_can.get_fdata(dtype=np.float32)
mean_epi = func_can_data.mean(axis=3)

# Canonical T1w также используем как float32.
anat_can_data = anat_can.get_fdata(dtype=np.float32)

# Registration terminology:
# static/fixed = reference image, которое оставляем в его пространстве;
# moving       = image, для которого ищем transformation.
static = anat_can_data
moving = mean_epi

static_affine = anat_can.affine
moving_affine = func_can.affine

print("Mean EPI / moving shape:", moving.shape)
print("T1w / static shape:", static.shape)

## 9. Initial alignment: centers of mass

Первый шаг — грубая инициализация.

Center-of-mass alignment выполняет только translation и даёт optimizеру стартовую точку.
Это ещё не полноценная rigid coregistration.

In [ ]:
from dipy.align.imaffine import transform_centers_of_mass

# Грубое начальное совмещение moving EPI с static T1w.
c_of_mass = transform_centers_of_mass(
    static,
    static_affine,
    moving,
    moving_affine,
)

# transform() применяет найденное преобразование и resamples moving image
# на voxel grid static image.
moving_com = c_of_mass.transform(moving)

print("Moving before:", moving.shape)
print("Static:", static.shape)
print("Moving after COM:", moving_com.shape)

In [ ]:
def normalize_for_display(data, low_percentile=2, high_percentile=98):
    """Robust normalization только для визуализации; исходные данные не изменяются."""
    low, high = np.percentile(data, [low_percentile, high_percentile])

    if high <= low:
        raise ValueError("Image has insufficient intensity range for display normalization.")

    clipped = np.clip(data, low, high)
    return (clipped - low) / (high - low)


static_norm = normalize_for_display(static)
moving_com_norm = normalize_for_display(moving_com)

In [ ]:
def plot_axial_overlay(static_image, moving_image, title, alpha=0.35):
    """Показывает central axial slice static, moving и их overlay."""
    z = static_image.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(static_image[:, :, z].T, cmap="gray", origin="lower")
    axes[0].set_title("T1w — static")

    axes[1].imshow(moving_image[:, :, z].T, cmap="gray", origin="lower")
    axes[1].set_title("Mean EPI — moving")

    axes[2].imshow(static_image[:, :, z].T, cmap="gray", origin="lower")
    axes[2].imshow(
        moving_image[:, :, z].T,
        cmap="magma",
        origin="lower",
        alpha=alpha,
    )
    axes[2].set_title(title)

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_axial_overlay(static_norm, moving_com_norm, "Overlay after COM")

После `transform()` output имеет T1w shape, но это **resampling**, а не увеличение реального
разрешения fMRI. Интерполяция не создаёт новых измеренных anatomical details.

## 10. Similarity metric: Mutual Information

T1w и EPI имеют разный intensity contrast, поэтому voxel-wise equality или обычная
линейная correlation — плохой критерий multimodal alignment.

**Mutual information (MI)** оценивает статистическую зависимость между intensity
distributions двух изображений.

При более согласованном anatomical alignment обычно возрастает MI.

`nbins=32` — это наша дискретизация интенсивностей для оценки joint/marginal
distributions, а не параметр MRI acquisition.

In [ ]:
from dipy.align.imaffine import AffineRegistration, MutualInformationMetric

# Используем все voxels для оценки mutual information.
metric = MutualInformationMetric(
    nbins=32,
    sampling_proportion=None,
)

# Multi-resolution optimization:
# coarse -> medium -> fine.
# Число iterations специально уменьшено относительно тяжёлого reference example,
# чтобы учебный notebook быстрее выполнялся в Colab.
level_iters = [1000, 100, 10]
sigmas = [3.0, 1.0, 0.0]
factors = [4, 2, 1]

affreg = AffineRegistration(
    metric=metric,
    level_iters=level_iters,
    sigmas=sigmas,
    factors=factors,
)

## 11. Translation optimization — 3 degrees of freedom

Сначала разрешаем optimizеру уточнить только `x/y/z` translation.

Стартуем не с identity, а с уже найденного center-of-mass transform.

In [ ]:
from dipy.align.transforms import TranslationTransform3D

translation = affreg.optimize(
    static,
    moving,
    TranslationTransform3D(),
    params0=None,
    static_grid2world=static_affine,
    moving_grid2world=moving_affine,
    starting_affine=c_of_mass.affine,
)

moving_translation = translation.transform(moving)
moving_translation_norm = normalize_for_display(moving_translation)

print("Translation affine:")
print(translation.affine)
print("\nRegistered shape:", moving_translation.shape)

## 12. Rigid coregistration — 6 degrees of freedom

Теперь разрешаем:

- 3 translations;
- 3 rotations.

Scaling и shear **не разрешаем**, потому что регистрируем изображения одного человека
и не хотим искусственно изменять размер / форму мозга.

In [ ]:
from dipy.align.transforms import RigidTransform3D

# Начальная точка для rigid optimization — уже оптимизированная translation.
rigid = affreg.optimize(
    static,
    moving,
    RigidTransform3D(),
    params0=None,
    static_grid2world=static_affine,
    moving_grid2world=moving_affine,
    starting_affine=translation.affine,
)

# Применяем итоговый rigid transform и resamples mean EPI в T1w grid.
moving_rigid = rigid.transform(moving)
moving_rigid_norm = normalize_for_display(moving_rigid)

print("Rigid affine:")
print(rigid.affine)
print("\nRegistered shape:", moving_rigid.shape)

В rigid affine:

- верхний левый `3×3` block кодирует rotation;
- последний столбец содержит translation component в world space;
- итоговый image всё равно остаётся resampled representation fMRI, а не «1 mm fMRI».

Небольшие off-diagonal значения в `3×3` block означают небольшой rotation.

## 13. Visual QC: translation vs rigid

Числовой optimization result нельзя принимать вслепую.  
Проверяем alignment визуально во всех трёх ортогональных плоскостях.

In [ ]:
def plot_registration_comparison(
    static_image,
    translation_image,
    rigid_image,
    alpha=0.35,
):
    """Сравнивает translation-only и rigid alignment в трёх плоскостях."""
    x, y, z = (np.array(static_image.shape) // 2).astype(int)

    slices = [
        ("Sagittal", (x, slice(None), slice(None))),
        ("Coronal", (slice(None), y, slice(None))),
        ("Axial", (slice(None), slice(None), z)),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    for col, (plane, slc) in enumerate(slices):
        # Верхний ряд: translation-only.
        axes[0, col].imshow(static_image[slc].T, cmap="gray", origin="lower")
        axes[0, col].imshow(
            translation_image[slc].T,
            cmap="magma",
            origin="lower",
            alpha=alpha,
        )
        axes[0, col].set_title(f"Translation — {plane}")

        # Нижний ряд: rigid.
        axes[1, col].imshow(static_image[slc].T, cmap="gray", origin="lower")
        axes[1, col].imshow(
            rigid_image[slc].T,
            cmap="magma",
            origin="lower",
            alpha=alpha,
        )
        axes[1, col].set_title(f"Rigid — {plane}")

    for ax in axes.ravel():
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_registration_comparison(
    static_norm,
    moving_translation_norm,
    moving_rigid_norm,
)

В наших данных rigid alignment визуально улучшает совпадение, особенно в coronal и axial views.

Остаточные локальные различия около orbitofrontal cortex нельзя автоматически считать
ошибкой registration: EPI там особенно чувствительна к susceptibility distortion и signal dropout.

## 14. Final contour QC

Overlay может скрывать часть T1w anatomy.  
Поэтому дополнительно рисуем intensity contours mean EPI поверх T1w.

Эти contour levels **не являются границами конкретных тканей**.  
Это лишь уровни нормализованной EPI intensity для визуальной проверки геометрии.

In [ ]:
def plot_contour_qc(static_image, moving_image, levels=(0.25, 0.5, 0.75)):
    """Contour-based QC rigid registration в sagittal, coronal и axial views."""
    x, y, z = (np.array(static_image.shape) // 2).astype(int)

    slices = [
        ("Sagittal", (x, slice(None), slice(None))),
        ("Coronal", (slice(None), y, slice(None))),
        ("Axial", (slice(None), slice(None), z)),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, (plane, slc) in zip(axes, slices):
        ax.imshow(static_image[slc].T, cmap="gray", origin="lower")
        ax.contour(
            moving_image[slc].T,
            levels=levels,
            origin="lower",
        )
        ax.set_title(f"Rigid QC — {plane}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_contour_qc(static_norm, moving_rigid_norm)

## 15. Итоги части I: пространственный checkpoint

Мы последовательно прошли:

`BIDS → NIfTI → voxel / volume / TR → affine → world coordinates → orientation → FOV → mean EPI → coregistration → resampling → visual QC`

Различаем:

- **Orientation** - направления осей массива относительно анатомических направлений.
- **Affine** - преобразование voxel indexes ↔ world coordinates.
- **Registration** - поиск преобразования для совмещения анатомического содержания изображений.
- **Resampling** - вычисление изображения на заданной voxel grid, не создание дополнительной измеренной детализации.
- **Coregistration** - здесь EPI и T1w одного участника.
- **Spatial normalization** - отдельный будущий этап: индивидуальное пространство → стандартное (например, MNI).

**Не выполнено:** motion correction, correction susceptibility distortions, brain masking, slice timing, normalization, smoothing и GLM. Это учебный пример, не production preprocessing; для дальнейшего исследования нужен валидированный preprocessing workflow (например, fMRIPrep).

---

# Часть II. Экспериментальные события, HRF и свёртка

Теперь переходим от пространственных изображений к временной структуре **того же BOLD run-01**. Эта часть не требует повторной загрузки большого NIfTI: переменная `func_img` уже содержит его shape и TR.

## 16. Чтение `events.tsv` и связь с fMRI

`onset` означает время с начала run (секунды), `duration` - длительность события (секунды). Стимулы и нажатия кнопок представлены отдельными строками. Нажатие кнопки не является новым предъявлением стимула.

Обрати внимание: времена стимулов **не обязаны совпадать** с границами томов при TR=2 s.

In [ ]:
# Используем тот же анонимный S3 client и ту же временную директорию,
# что в части I. Не скачиваем большой BOLD-файл повторно.
events_key = (
    "ds000117/sub-01/ses-mri/func/"
    "sub-01_ses-mri_task-facerecognition_run-01_events.tsv"
)
events_path = DATA_DIR / "sub-01_run-01_events.tsv"

if not events_path.exists():
    s3.download_file(bucket, events_key, str(events_path))

events = pd.read_csv(events_path, sep="\t")

tr = float(func_img.header.get_zooms()[3])
n_volumes = func_img.shape[3]
run_duration = tr * n_volumes

print(f"Volumes: {n_volumes}; TR: {tr:g} s; run duration: {run_duration:g} s")
print(f"Event rows: {len(events)}; last onset: {events['onset'].max():.3f} s")
display(events.head(10))

## 17. Декодируем экспериментальные условия

Участник выполнял задачу с изображениями лиц и scrambled-стимулами. В обозначениях `trial_type` зашиты **категория стимула** и **тип предъявления (repetition)**:

| Код | Условие |
|---|---|
| `IniFF` / `ImmFF` / `DelFF` | Famous faces: initial / immediate / delayed |
| `IniUF` / `ImmUF` / `DelUF` | Unfamiliar faces: initial / immediate / delayed |
| `IniSF` / `ImmSF` / `DelSF` | Scrambled stimuli: initial / immediate / delayed |
| `btnL`, `btnR` | Нажатия левой / правой кнопки, **не стимулы** |

Итого: 3 категории × 3 типа предъявления = **9 условий**.

`response_time` на строках с нажатиями относится к времени реакции на соответствующий стимул, а не к абсолютному времени от начала run: для пары `ImmFF` (onset 3.273 s) → `btnL` (onset 4.506 s) разность равна 1.233 s.

Для первой модели `faces vs scrambled` мы объединим famous и unfamiliar, временно игнорируя различия repetition. Это **аналитическое решение**, а не свойство измеренного сигнала.

In [ ]:
# Явно выбираем только стимульные коды, чтобы случайно не принять
# поведенческие ответы и другие возможные события за предъявления.
stimulus_mask = events["trial_type"].str.fullmatch(
    r"(?:Ini|Imm|Del)(?:FF|UF|SF)", na=False
)
stimulus_events = events.loc[stimulus_mask].copy()
response_events = events.loc[events["trial_type"].isin(["btnL", "btnR"])].copy()

# Две независимые оси экспериментального дизайна.
repetition_map = {"Ini": "initial", "Imm": "immediate", "Del": "delayed"}
category_map = {"FF": "famous", "UF": "unfamiliar", "SF": "scrambled"}

stimulus_events["repetition"] = stimulus_events["trial_type"].str[:3].map(repetition_map)
stimulus_events["category"] = stimulus_events["trial_type"].str[-2:].map(category_map)
stimulus_events["analysis_class"] = stimulus_events["category"].replace(
    {"famous": "face", "unfamiliar": "face", "scrambled": "scrambled"}
)

print(f"Stimulus events: {len(stimulus_events)}")
print(f"Button responses: {len(response_events)}")
print("\nAnalysis classes:")
print(stimulus_events["analysis_class"].value_counts().to_string())

# Контроль целостности: все 93 экспериментальных стимула распознаны.
assert len(stimulus_events) == 93
assert len(stimulus_events) + len(response_events) == len(events)

### Проверяем баланс девяти условий

Таблица показывает, сколько стимулов каждого типа было в run-01. Количества близки, но **не одинаковы**; эти различия будут влиять на высоты предикторов, даже если всем событиям задать вес 1.

In [ ]:
condition_counts = pd.crosstab(
    stimulus_events["repetition"], stimulus_events["category"]
)
display(condition_counts)

## 18. Timeline экспериментальных событий

По оси X - **реальные onsets из `events.tsv`**. Цвет означает категорию изображения, форма маркера - initial / immediate / delayed. Мы оставляем девять рядов: так видна структура эксперимента без наложения всех точек на одну линию. `duration` пока не отображаем.

In [ ]:
# Девять экспериментальных условий: три категории × три типа предъявления.
conditions = [
    (category, repetition)
    for category in ("famous", "unfamiliar", "scrambled")
    for repetition in ("initial", "immediate", "delayed")
]
y_positions = [10, 9, 8, 6, 5, 4, 2, 1, 0]
colors = {"famous": "C0", "unfamiliar": "C1", "scrambled": "C2"}
markers = {"initial": "o", "immediate": "x", "delayed": "^"}

fig, ax = plt.subplots(figsize=(16, 7))
for (category, repetition), y in zip(conditions, y_positions):
    subset = stimulus_events.loc[
        stimulus_events["category"].eq(category)
        & stimulus_events["repetition"].eq(repetition)
    ]
    ax.plot(
        subset["onset"], [y] * len(subset),
        linestyle="None", marker=markers[repetition],
        color=colors[category], markerfacecolor="none",
        markersize=7, markeredgewidth=1.6,
    )

ax.set_yticks(
    y_positions,
    [f"{category.title()} - {repetition.title()}" for category, repetition in conditions],
)
for boundary in (7, 3):
    ax.axhline(boundary, color="gray", linestyle="--", alpha=0.4)

# Цвет различает категории (видно на графике), форма - repetition (легенда).
legend_handles = [
    Line2D([0], [0], marker=marker, linestyle="None",
           markerfacecolor="none", markeredgecolor="black",
           markersize=8, label=name.title())
    for name, marker in markers.items()
]
ax.legend(handles=legend_handles, title="Repetition")
ax.set_xlim(0, run_duration)
ax.set_ylim(-0.7, 10.7)
ax.set_xlabel("Time from run start (s)")
ax.set_title("Wakeman & Henson - Experimental event timeline\nsub-01 | run-01")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

**Saved teaching result (sub-01, run-01):**

![9 conditions: stimulus timeline](../figures/results/experimental_event_timeline_run01.png)

This image shows experiment timing or **modeled** HRF predictors, not statistical brain activation.


## 19. От события к ожидаемому BOLD: canonical HRF

BOLD - это **косвенный**, зависящий от гемодинамики MR-показатель, а не запись активности отдельных нейронов. Изменения нейронной активности могут вызвать neurovascular coupling, изменения кровотока и относительной концентрации deoxyhemoglobin, а затем изменения T2*-взвешенного сигнала.

Первое учебное изображение иллюстрирует лишь один аспект положительного BOLD: уменьшение относительного вклада deoxygenated hemoglobin при сосудистом ответе. **Это не количественная модель CBF или CMRO₂ и не изображение, полученное у sub-01.**

**Схема механизма (словами):** neuronal activity → neurovascular coupling → изменения кровотока, объёма крови и oxygenation → T2*-взвешенное изменение BOLD. Это косвенный показатель, а не прямое считывание спайков или нейронных потенциалов.


**Hemodynamic response function (HRF)** - функция, связывающая момент модельного события и ожидаемую временную форму BOLD-ответа. Типичная положительная HRF достигает максимума через несколько секунд, затем снижается и может иметь *post-stimulus undershoot*.

Мы используем **canonical SPM HRF**: общую форму, принятую для расчётов, а не индивидуально измеренную реакцию. Предполагаем приблизительно одинаковую форму для событий и линейную суперпозицию ответов. Реальные HRF могут различаться по регионам и участникам.

**Форма canonical HRF:** событие → задержанный положительный пик примерно через несколько секунд → спад → возможный post-stimulus undershoot. Индивидуальная форма у участника здесь не измерялась.


### Два реальных onsets и два модельных ответа

Возьмём первые `IniUF` (6.647 s) и `ImmUF` (9.838 s). Для визуализации считаем события мгновенными и нормируем HRF к максимуму 1. Момент пика **относителен к onset**, поэтому пик каждого ответа появляется спустя несколько секунд после стимула.

In [ ]:
# Первый initial/unfamiliar и его ближайшее immediate-повторение в run-01.
t1 = float(stimulus_events.loc[stimulus_events["trial_type"].eq("IniUF"), "onset"].iloc[0])
t2 = float(stimulus_events.loc[stimulus_events["trial_type"].eq("ImmUF"), "onset"].iloc[0])

# Десять вычислительных точек на секунду. t_r здесь выбран для генерации
# HRF с шагом 0.1 s; РЕАЛЬНЫЙ TR сканера равен 2 s и не меняется.
example_dt = 0.1
example_time = np.arange(0, 35, example_dt)
hrf_example = spm_hrf(t_r=1.0, oversampling=10)
hrf_sample_time = np.arange(len(hrf_example)) * example_dt
hrf_example = hrf_example / hrf_example.max()

# Сдвигаем одинаковую HRF к началу каждого события (амплитуды по 1).
response1 = np.interp(example_time - t1, hrf_sample_time, hrf_example, left=0, right=0)
response2 = np.interp(example_time - t2, hrf_sample_time, hrf_example, left=0, right=0)
combined = response1 + response2

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(example_time, response1, label="IniUF response")
ax.plot(example_time, response2, label="ImmUF response")
ax.plot(example_time, combined, "--", linewidth=2, label="Combined response")
for onset in (t1, t2):
    ax.axvline(onset, linestyle=":", alpha=0.5)
ax.set(xlabel="Time from run start (s)", ylabel="Relative model response",
       title="Overlapping HRFs - ds000117 stimulus onsets")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Что меняется, если второй ответ слабее?

Ни onset, ни HRF не изменяем. Снижаем лишь **заданный нами вес** второго события до 0.5. Это позволяет понять влияние амплитуды на суммарный максимум и время его достижения.

Важно: снижение до 0.5 **не измерено** у участника и **не доказывает repetition suppression**.

In [ ]:
# Учебная гипотеза: второй ответ вдвое слабее при неизменной форме HRF.
response2_weak = 0.5 * response2
combined_weak = response1 + response2_weak

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(example_time, response1, label="IniUF: amplitude 1")
ax.plot(example_time, response2_weak, label="ImmUF: amplitude 0.5")
ax.plot(example_time, combined_weak, "--", linewidth=2.5, label="Combined response")
for onset in (t1, t2):
    ax.axvline(onset, linestyle=":", alpha=0.4)
ax.set(xlabel="Time from run start (s)", ylabel="Relative model response",
       title="Two overlapping HRFs with different amplitudes")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 20. Свёртка (convolution): от двух HRF к общей временной кривой

Для одного события в момент `t_i` ответ имеет вид `h(t − t_i)`. Для серии событий сумма сдвинутых ответов вычисляется свёрткой:

\[x[n] = (s*h)[n] = \sum_k s[k]\, h[n-k]\]

- `s[k]` - вес события в момент `k`; в нашем учебном примере это 0 или 1.
- `h[n−k]` - вклад HRF спустя `n−k` временных шагов после события.
- `x[n]` - ожидаемая сумма ответов в момент `n`.

**Ручной пример:** события в секунды 1 и 3, искусственная HRF `h=[0,1,2,2,1]`. При `n=4`: `x[4]=h[3]+h[1]=2+1=3`. При `n=6`: `x[6]=h[5]+h[3]=0+2=2` (значения HRF вне заданного диапазона в математической свёртке считаются нулевыми; прямое обращение `h[5]` в Python вызвало бы ошибку).

График ниже - **только иллюстрация дискретного примера**, не реальные данные fMRI.

**Дискретный контроль:** `s=[0,1,0,1,0,0,0,0]`, `h=[0,1,2,2,1]` → `x=[0,0,1,2,3,3,2,1]`. Для `n=4`: `h[3]+h[1]=2+1=3`. Это модель, не нейронные данные.


In [ ]:
# Сверяем ручной расчёт со встроенной дискретной свёрткой NumPy.
toy_events = np.array([0, 1, 0, 1, 0, 0, 0, 0])
toy_hrf = np.array([0, 1, 2, 2, 1])
toy_predictor = np.convolve(toy_events, toy_hrf, mode="full")[:len(toy_events)]

print("Events:     ", toy_events)
print("HRF:        ", toy_hrf)
print("Convolution:", toy_predictor)
assert np.array_equal(toy_predictor, [0, 0, 1, 2, 3, 3, 2, 1])

### Два реальных события через `np.convolve()`

Повторяем уже знакомые onsets, но теперь не строим и не суммируем HRF вручную: сначала создаём массив событий с мелким шагом 0.1 s, затем вызываем свёртку. Округление onset до ближайшей точки временной сетки даёт погрешность до 0.05 s.

Если вес повторного события изменить с 1.0 на 0.5, кривая станет ниже, а пик может сместиться к первому ответу.

In [ ]:
# Отдельные имена переменных не позволяют учебной свёртке перезаписать
# массивы из раздела 19 или будущий общий предиктор всего run.
example_train = np.zeros(len(example_time))
repeat_weight = 1.0  # Можно временно изменить на 0.5 для сравнения.
example_train[round(t1 / example_dt)] = 1.0
example_train[round(t2 / example_dt)] = repeat_weight

example_convolved = np.convolve(example_train, hrf_example, mode="full")[:len(example_time)]

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(example_time, example_convolved, label="Convolved BOLD predictor")
for onset in (t1, t2):
    ax.axvline(onset, linestyle=":", alpha=0.5)
ax.set(xlabel="Time from run start (s)", ylabel="Relative model response",
       title="Two event impulses convolved with canonical HRF")
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

## 21. Все 93 события → общий предиктор

Теперь переносим вычисления на весь `run-01` (416 s), используя fine grid 0.1 s. Каждое из **93 предъявлений** получает одинаковый вес 1. Текущая модель рассматривает стимулы как импульсы (onset-only), несмотря на ненулевые `duration` в `events.tsv`; их учёт - отдельный следующий шаг.

Функция `make_event_train()` пригодится трижды (все стимулы, faces, scrambled). `np.add.at` корректно суммирует события, даже если два попадут в один временной bin.

In [ ]:
# Единая функция вместо трёх повторных блоков создания event train.
def make_event_train(onsets, dt, n_time_points):
    """Convert onset times (s) into a discrete unit-amplitude event train."""
    event_indices = np.rint(np.asarray(onsets, dtype=float) / dt).astype(int)
    if np.any((event_indices < 0) | (event_indices >= n_time_points)):
        raise ValueError("At least one onset is outside the model time grid")
    train = np.zeros(n_time_points, dtype=float)
    np.add.at(train, event_indices, 1.0)
    return train

# Это вычислительная сетка, НЕ частота выборки fMRI.
dt = 0.1
time = np.arange(0, run_duration, dt)

# Canonical SPM HRF: max=1 только для иллюстративного масштаба предикторов.
hrf = spm_hrf(t_r=1.0, oversampling=10)
hrf = hrf / hrf.max()

event_train = make_event_train(stimulus_events["onset"], dt, len(time))
predictor = np.convolve(event_train, hrf, mode="full")[:len(time)]

print("Number of stimuli:", len(stimulus_events))
print("Number of model time points:", len(time))
print("Scanner volumes:", n_volumes)

### Визуализация: event train и суммарный HRF-предиктор

Верхний график - реальные времена стимулов. Нижний - **предсказанная форма** по модели, а не записанный сканером BOLD. Мелкие «зубцы» внутри плато возникают из-за наложения HRF от близко расположенных событий. Более высокие уровни отражают плотность и расположение стимулов, а не доказанную «сильную активацию». Отрицательные провалы - часть выбранной canonical HRF.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

ax[0].vlines(
    stimulus_events["onset"], 0, 1,
    color="gray", linewidth=0.8
)
ax[0].set_ylabel("Stimulus events")
ax[0].set_title("Actual stimulus onsets")

ax[1].plot(time, predictor)
ax[1].set_ylabel("Relative model response")
ax[1].set_xlabel("Time from run start (s)")
ax[1].set_title("HRF-convolved event train")
ax[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

**Saved teaching result (sub-01, run-01):**

![All stimulus events and their HRF convolution](../figures/results/stimulus_hrf_convolution_run01.png)

This image shows experiment timing or **modeled** HRF predictors, not statistical brain activation.


## 22. Два отдельных предиктора: `faces` vs `scrambled`

Если сложить все категории в одну последовательность, мы теряем информацию о том, какой стимул вызвал каждый предполагаемый ответ. Поэтому выделяем две группы, соответствующие первой запланированной исследовательской задаче:

- **Faces:** 31 famous + 30 unfamiliar = 61 предъявление.
- **Scrambled:** 32 предъявления.

Пока каждой группе задаём одинаковую canonical HRF и вес события 1. **Различия между предикторами отражают расписание событий, а не найденные эффекты мозга.**

In [ ]:
# Два условия из заранее созданной analysis_class.
faces = stimulus_events.loc[stimulus_events["analysis_class"].eq("face")].copy()
scrambled = stimulus_events.loc[stimulus_events["analysis_class"].eq("scrambled")].copy()

print("Faces:", len(faces))
print("Scrambled:", len(scrambled))
print("Total:", len(faces) + len(scrambled))
assert len(faces) + len(scrambled) == len(stimulus_events)

In [ ]:
# Для двух категорий применяем ту же временную сетку и ту же HRF.
face_train = make_event_train(faces["onset"], dt, len(time))
scrambled_train = make_event_train(scrambled["onset"], dt, len(time))

face_predictor = np.convolve(face_train, hrf, mode="full")[:len(time)]
scrambled_predictor = np.convolve(scrambled_train, hrf, mode="full")[:len(time)]

# Проверка линейности свёртки: разделение по категориям не должно
# изменять общий предиктор (при одинаковой HRF и единичных весах).
reconstructed = face_predictor + scrambled_predictor
print("Predictors match:", np.allclose(reconstructed, predictor))
assert np.allclose(reconstructed, predictor)

### Сравнение будущих регрессоров

Два графика ещё **не описывают различия в BOLD-активации на лица и scrambled**. Они показывают, **когда модель ожидала бы гемодинамический ответ**, если бы все предъявления вызывали одинаковый импульсный эффект. Их частичное совпадение во времени позже важно для интерпретации коэффициентов GLM.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(time, face_predictor, label=f"Faces ({len(faces)} stimuli)", linewidth=1.8)
ax.plot(time, scrambled_predictor, label=f"Scrambled ({len(scrambled)} stimuli)", linewidth=1.8)
ax.set(title="HRF-convolved predictors: Faces vs Scrambled",
       xlabel="Time from run start (s)", ylabel="Relative model response",
       xlim=(0, run_duration))
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Saved teaching result (sub-01, run-01):**

![Model predictors for faces and scrambled stimuli](../figures/results/faces_vs_scrambled_hrf_predictors_run01.png)

This image shows experiment timing or **modeled** HRF predictors, not statistical brain activation.


## 23. Итоги части II и ограничения

**Что проверено на реальных данных `ds000117`:**

- прочитана временная структура одного run (208 volumes, TR=2 s, 416 s);
- отделены 93 стимула от 92 нажатий кнопок;
- разобраны 9 сочетаний категории × repetition;
- визуализированы времена событий;
- исследовано сложение двух сдвинутых HRF и влияние гипотетических амплитуд;
- ручная дискретная свёртка проверена через `np.convolve()`;
- построен ожидаемый гемодинамический предиктор из всех 93 onsets;
- отдельно построены `faces` и `scrambled` и проверено: `faces + scrambled = all events` (в пределах численной точности).

**Это пока не результаты fMRI-статистики.** Нет регрессии, оценённых beta, теста `faces > scrambled`, вывода о repetition suppression или локализации активности. Условные веса 1 и 0.5 - **не нейронные параметры**, полученные из измерений.

**Рабочие упрощения:**

1. Только `onset`, без учёта реальной `duration` (временная модель импульсных событий).
2. Одна canonical HRF для всех событий; линейная суперпозиция считается приближением.
3. Временная сетка 0.1 s и округление onsets до ближайшего bin; это **не TR=2 s**, по которому фактически записаны fMRI volumes.
4. Переменные `faces` и `scrambled` могут иметь пересекающиеся временные профили, что нужно учитывать при статистической оценке эффектов.
5. Работа с raw BOLD в части I не заменяет fMRI preprocessing.

### Следующий исследовательский шаг

**`event durations → sampling regressors at fMRI acquisition times → design matrix`**. Сначала разберём, как учесть длительности стимулов и перейти от fine grid к 208 fMRI time points. Затем - GLM для реального (предобработанного) BOLD.

*Project status: пространственные основы закрыты; events/HRF/convolution изучены и задокументированы; дизайн-матрица ещё не построена.*